# AI Hedge Fund — LSTM Dual-Head Training

In [ ]:
!pip install pyarrow tensorflow --quiet
print('Dependencies ready.')

## Cell 2 — Imports

In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
import gc
import warnings
from datetime import datetime

from scipy.stats import spearmanr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks

warnings.filterwarnings('ignore')
print(f'TensorFlow   : {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

for _gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(_gpu, True)
print('GPU memory growth enabled.')

## Cell 3 — Paths

In [ ]:
DRIVE_BASE   = '/kaggle/working'

RESULTS_DIR  = os.path.join(DRIVE_BASE, 'results', 'lstm')
MODELS_DIR   = os.path.join(DRIVE_BASE, 'models')
PARQUET_PATH = "/kaggle/input/datasets/loc0487/hedge-fund-features/features_master_latest.parquet"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)

DATE_STAMP = datetime.today().strftime('%Y%m%d')
print(f'Date stamp  : {DATE_STAMP}')
print(f'Parquet     : {PARQUET_PATH}')
print(f'Results dir : {RESULTS_DIR}')
print(f'Models dir  : {MODELS_DIR}')

## Cell 4 — Config

In [ ]:
SEQ_LEN        = 60
MIN_CLEAN_DAYS = 500

TRAIN_END_DATE = pd.Timestamp('2023-12-31')
BACKTEST_START = pd.Timestamp('2024-01-01')

TEST_MONTHS    = 6
N_FOLDS        = 8

MAX_TRAIN_YEARS = 4

BATCH_SIZE    = 512
EPOCHS        = 100
PATIENCE      = 25
LEARNING_RATE = 0.001

TARGET_RANK = 'Target_Return_21d'
TARGET_VOL  = 'Target_Vol_5d'
DATE_COL    = 'Date'
TICKER_COL  = 'Ticker'
TIER_COL    = 'Data_Tier'

LOSS_WEIGHT_RANK = 0.7
LOSS_WEIGHT_VOL  = 0.3
CS_MIN_STOCKS    = 10
USE_REGIME_INT   = True

print('Config loaded.')
print(f'  SEQ_LEN          : {SEQ_LEN}')
print(f'  TRAIN_END_DATE   : {TRAIN_END_DATE.date()}')
print(f'  BACKTEST_START   : {BACKTEST_START.date()}')
print(f'  MAX_TRAIN_YEARS  : {MAX_TRAIN_YEARS}  (sliding window — None = anchored expanding)')
print(f'  CV               : sliding {MAX_TRAIN_YEARS}yr / {TEST_MONTHS}mo test / {N_FOLDS} folds')
print(f'  USE_REGIME_INT   : {USE_REGIME_INT}')

## Cell 5 — Load + Filter Data

In [ ]:
print('Loading Parquet ...')
df = pd.read_parquet(PARQUET_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
print(f'Raw rows        : {len(df):,}')

df = df[df[TIER_COL] == 1].copy()
print(f'After Tier A    : {len(df):,}  ({df[TICKER_COL].nunique()} tickers)')

non_feature_cols = {
    DATE_COL, TICKER_COL, TIER_COL, TARGET_RANK, TARGET_VOL,
    'Target_Return_21d', 'Target_Direction_Median',
    'Target_Direction_Tertile', 'FII_Source_Flag', 'id',
}
for col in df.columns:
    if col not in non_feature_cols and df[col].dtype == object:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.sort_values([TICKER_COL, DATE_COL]).reset_index(drop=True)
print(f'Full date range : {df[DATE_COL].min().date()} -> {df[DATE_COL].max().date()}')

df_train_pool = df[df[DATE_COL] <= TRAIN_END_DATE].copy()
df_backtest   = df[df[DATE_COL] >= BACKTEST_START].copy()
print(f'Training pool   : {df_train_pool[DATE_COL].min().date()} -> '
      f'{df_train_pool[DATE_COL].max().date()}  ({len(df_train_pool):,} rows)')
print(f'Backtest pool   : {df_backtest[DATE_COL].min().date()} -> '
      f'{df_backtest[DATE_COL].max().date()}  ({len(df_backtest):,} rows)  <- HELD-OUT')

## Cell 6 — Define LSTM Feature Columns

In [ ]:
RETURN_COLS = [
    'Log_Return_1d', 'Return_5d', 'Return_21d', 'Return_60d',
    'Volatility_20d', 'Volume_Ratio_20d',
]
RS_COLS = ['RS_21d', 'RS_60d']
RATIO_COLS = [
    'Price_to_SMA20', 'Price_to_SMA50', 'Price_to_SMA200',
    'Price_to_52W_High', 'Price_to_52W_Low',
]
TECH_COLS = [
    'MACD_Hist', 'RSI_14', 'BB_Width', 'BB_PctB', 'ATR_14',
    'Stoch_K', 'ADX_14', 'OBV_Change_5d',
    'VWAP_Dev',
]
REGIME_COLS = ['Regime_Int'] if USE_REGIME_INT else []
BETA_COLS = [
    'Beta_to_Nifty50', 'Beta_to_Nifty500', 'Beta_to_USDINR',
    'Beta_to_VIX', 'Beta_to_Crude', 'Beta_to_Gold',
    'Beta_to_FII', 'Beta_to_DII', 'Repo_Rate_Change',
]

FILL_VALUES = {
    'Price_to_SMA20': 1.0, 'Price_to_SMA50': 1.0, 'Price_to_SMA200': 1.0,
    'Price_to_52W_High': 1.0, 'Price_to_52W_Low': 1.0,
    'Beta_to_Nifty50': 0.0, 'Beta_to_Nifty500': 0.0, 'Beta_to_USDINR': 0.0,
    'Beta_to_VIX': 0.0, 'Beta_to_Crude': 0.0, 'Beta_to_Gold': 0.0,
    'Beta_to_FII': 0.0, 'Beta_to_DII': 0.0, 'Repo_Rate_Change': 0.0,
    'Regime_Int': 0.0, 'OBV_Change_5d': 0.0, 'ATR_14': 0.0,
}
DEFAULT_FILL = 0.0

_CANDIDATE_COLS = RETURN_COLS + RS_COLS + RATIO_COLS + TECH_COLS + REGIME_COLS + BETA_COLS
LSTM_FEATURE_COLS = [
    c for c in _CANDIDATE_COLS
    if c in df.columns and pd.api.types.is_numeric_dtype(df[c])
]
missing_cols = [c for c in _CANDIDATE_COLS if c not in df.columns]

print(f'LSTM features   : {len(LSTM_FEATURE_COLS)}  (candidate: {len(_CANDIDATE_COLS)})')
if missing_cols:
    print(f'[WARN] Missing from Parquet: {missing_cols}')
else:
    print('[OK] All candidate columns present.')
print(f'  Columns: {LSTM_FEATURE_COLS}')

## Cell 7 — Imputation Helper

In [ ]:
def impute_features(df_in, feature_cols, fill_values, default_fill=0.0):
    df_out = df_in.copy()

    df_out[feature_cols] = (
        df_out.groupby(TICKER_COL, sort=False)[feature_cols]
              .transform(lambda s: s.ffill().bfill())
    )

    for col in feature_cols:
        if df_out[col].isna().any():
            df_out[col] = df_out[col].fillna(fill_values.get(col, default_fill))

    return df_out

print('Imputation helper defined (vectorized groupby+transform).')

## Cell 8 — Cross-Sectional Normalizer

In [ ]:
class CrossSectionalNormalizer:
    def __init__(self, feature_cols, min_stocks=10):
        self.feature_cols = feature_cols
        self.min_stocks   = min_stocks
        self.stats_       = None

    def fit(self, df_wide):
        grp    = df_wide.groupby(DATE_COL)[self.feature_cols]
        counts = df_wide.groupby(DATE_COL)[TICKER_COL].nunique()
        valid  = counts[counts >= self.min_stocks].index

        means = grp.mean().loc[valid]
        stds  = grp.std().loc[valid]

        means.columns = [f'mean_{c}' for c in means.columns]
        stds.columns  = [f'std_{c}'  for c in stds.columns]

        self.stats_ = pd.concat([means, stds], axis=1).sort_index()
        print(f'  Normalizer fitted on {len(self.stats_):,} training dates')
        return self

    def transform(self, df_wide):
        if self.stats_ is None:
            raise RuntimeError('Call fit() before transform().')

        df_out = df_wide.copy()

        all_dates = pd.DatetimeIndex(
            sorted(set(self.stats_.index.tolist()) |
                   set(pd.to_datetime(df_out[DATE_COL]).unique()))
        )
        stats_ff = self.stats_.reindex(all_dates).ffill()

        row_dates  = pd.to_datetime(df_out[DATE_COL]).values
        date_index = stats_ff.index.values
        pos        = np.searchsorted(date_index, row_dates, side='right') - 1
        pos        = np.clip(pos, 0, len(date_index) - 1)

        valid_features = [f for f in self.feature_cols
                          if f'mean_{f}' in stats_ff.columns]
        if not valid_features:
            return df_out

        mean_mat  = stats_ff[[f'mean_{f}' for f in valid_features]].values[pos]
        std_mat   = stats_ff[[f'std_{f}'  for f in valid_features]].values[pos]
        feat_mat  = df_out[valid_features].values.astype(np.float64)
        std_safe  = np.where((std_mat == 0) | np.isnan(std_mat), np.nan, std_mat)
        normed    = (feat_mat - mean_mat) / std_safe
        normed    = np.where(np.isnan(normed), feat_mat, normed)

        df_out[valid_features] = normed
        return df_out

    def save(self, path):
        with open(path, 'wb') as f:
            pickle.dump(self, f)

    @staticmethod
    def load(path):
        with open(path, 'rb') as f:
            return pickle.load(f)

print('CrossSectionalNormalizer defined.')

## Cell 9 — Filter Tickers with Sufficient History

In [ ]:
print('Counting usable rows per ticker ...')

df_pool_imp = impute_features(
    df_train_pool.sort_values([TICKER_COL, DATE_COL]),
    LSTM_FEATURE_COLS, FILL_VALUES, DEFAULT_FILL
)
ticker_counts = (
    df_pool_imp.dropna(subset=[TARGET_RANK, TARGET_VOL])
               .groupby(TICKER_COL)
               .size()
)
del df_pool_imp
gc.collect()

eligible = ticker_counts[ticker_counts >= MIN_CLEAN_DAYS].index.tolist()
excluded = ticker_counts[ticker_counts  < MIN_CLEAN_DAYS].index.tolist()

print(f'Tickers with >= {MIN_CLEAN_DAYS} usable days : {len(eligible)}')
print(f'Tickers excluded (insufficient history)     : {len(excluded)}')
if excluded:
    print(f'  {excluded[:10]}{"..." if len(excluded) > 10 else ""}')

df_lstm = df_train_pool[df_train_pool[TICKER_COL].isin(eligible)].copy()
tickers = sorted(df_lstm[TICKER_COL].unique().tolist())

print(f'Rows for LSTM training : {len(df_lstm):,}')
print(f'Tickers                : {len(tickers)}')
print(f'Date range (train)     : {df_lstm[DATE_COL].min().date()} -> '
      f'{df_lstm[DATE_COL].max().date()}')

## Cell 10 — Sequence Builders

In [ ]:
def _build_sequences_lists(df_norm, tickers_list, feature_cols, seq_len):
    all_X, all_yr, all_yv, all_dates = [], [], [], []

    grouped = {k: v.sort_values(DATE_COL).reset_index(drop=True)
               for k, v in df_norm.groupby(TICKER_COL)}

    for ticker in tickers_list:
        t_df = grouped.get(ticker)
        if t_df is None or len(t_df) < seq_len + 1:
            continue
        feat_vals = t_df[feature_cols].values.astype(np.float32)
        for i in range(seq_len, len(t_df)):
            r = t_df.at[i, TARGET_RANK]
            v = t_df.at[i, TARGET_VOL]
            if pd.isna(r) or pd.isna(v):
                continue
            all_X.append(feat_vals[i - seq_len:i])
            all_yr.append(float(r))
            all_yv.append(float(v))
            all_dates.append(t_df.at[i, DATE_COL])
    return all_X, all_yr, all_yv, all_dates

def build_train_dataset(df_norm, tickers_list, feature_cols, seq_len, batch_size):
    all_X, all_yr, all_yv, _ = _build_sequences_lists(
        df_norm, tickers_list, feature_cols, seq_len
    )
    if not all_X:
        return None, 0, None, None, None

    X_arr  = np.array(all_X,  dtype=np.float32)
    yr_arr = np.array(all_yr, dtype=np.float32)
    yv_arr = np.array(all_yv, dtype=np.float32)
    n      = len(X_arr)

    shuffle_buf = min(10000, n)
    ds = (
        tf.data.Dataset.from_tensor_slices(
            (X_arr, {'rank_output': yr_arr, 'vol_output': yv_arr})
        )
        .shuffle(buffer_size=shuffle_buf, reshuffle_each_iteration=True)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )
    return ds, n, X_arr, yr_arr, yv_arr

def build_test_sequences(df_norm, tickers_list, feature_cols, seq_len):
    """Numpy arrays for test/backtest. Test sets ~18-25K rows — safe in memory."""
    all_X, all_yr, all_yv, all_dates = _build_sequences_lists(
        df_norm, tickers_list, feature_cols, seq_len
    )
    if not all_X:
        return None, None, None, None
    return (
        np.array(all_X,      dtype=np.float32),
        np.array(all_yr,     dtype=np.float32),
        np.array(all_yv,     dtype=np.float32),
        np.array(all_dates),
    )

print('Sequence builders defined')


## Cell 11 — Build LSTM Model

In [ ]:
def build_lstm_model(seq_len, n_features, learning_rate,
                     loss_weight_rank, loss_weight_vol):
    inp = keras.Input(shape=(seq_len, n_features), name='sequence_input')
    x = layers.LSTM(128, return_sequences=True,  name='lstm_1')(inp)
    x = layers.Dropout(0.2, name='dropout_1')(x)
    x = layers.LSTM(64,  return_sequences=False, name='lstm_2')(x)
    x = layers.Dropout(0.2, name='dropout_2')(x)
    x = layers.Dense(32, activation='relu', name='dense_shared')(x)
    head_rank = layers.Dense(1, activation='tanh', name='rank_output')(x)
    head_vol  = layers.Dense(1, activation='softplus', name='vol_output')(x)
    model = Model(inputs=inp, outputs=[head_rank, head_vol],
                  name='lstm_dual_head_v4')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=1.0),
        loss={
            'rank_output': 'mse',
            'vol_output' : 'mse',
        },
        loss_weights={
            'rank_output': loss_weight_rank,
            'vol_output' : loss_weight_vol,
        },
        metrics={
            'rank_output': ['mae'],
            'vol_output' : ['mae'],
        }
    )
    return model

_tmp = build_lstm_model(SEQ_LEN, len(LSTM_FEATURE_COLS),
                        LEARNING_RATE, LOSS_WEIGHT_RANK, LOSS_WEIGHT_VOL)
_tmp.summary()
del _tmp
keras.backend.clear_session()

## Cell 12 — IC Helper Functions

In [ ]:
def compute_ic_cross_sectional(y_true, y_pred, dates):
    """Per-date Spearman IC, averaged across dates. Correct metric for rank target."""
    df_ic = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred, DATE_COL: dates})
    daily = []
    for date, grp in df_ic.groupby(DATE_COL):
        if len(grp) < 5:
            continue
        ic, _ = spearmanr(grp['y_true'], grp['y_pred'])
        if not np.isnan(ic):
            daily.append({DATE_COL: date, 'IC': float(ic)})
    if not daily:
        return 0.0, 0.0, pd.Series(dtype=float)
    ic_series = pd.DataFrame(daily).set_index(DATE_COL)['IC']
    mean_ic   = float(ic_series.mean())
    icir      = float(mean_ic / ic_series.std()) if ic_series.std() > 0 else 0.0
    return mean_ic, icir, ic_series

def compute_ic_pooled(y_true, y_pred):
    """Pooled Spearman IC — for vol head."""
    ic, _ = spearmanr(y_true, y_pred)
    return float(ic) if not np.isnan(ic) else 0.0

print('IC helpers defined.')

## Cell 13 — Build Walk-Forward Folds (Anchored Expanding)

In [ ]:
from dateutil.relativedelta import relativedelta

data_start = df_lstm[DATE_COL].min()
data_end   = df_lstm[DATE_COL].max()

print(f'Training pool range : {data_start.date()} -> {data_end.date()}')
assert data_end <= TRAIN_END_DATE, (
    f'data_end {data_end.date()} exceeds TRAIN_END_DATE {TRAIN_END_DATE.date()}'
)

folds = []
for i in range(N_FOLDS):
    test_end   = data_end  - relativedelta(months=TEST_MONTHS * i)
    test_start = test_end  - relativedelta(months=TEST_MONTHS)
    train_end  = test_start

    if MAX_TRAIN_YEARS is not None:
        train_start = max(data_start,
                         train_end - relativedelta(years=MAX_TRAIN_YEARS))
    else:
        train_start = data_start

    if (train_end - train_start).days < SEQ_LEN + MIN_CLEAN_DAYS:
        print(f'  Fold {i+1}: insufficient training span — stopping.')
        break

    folds.append({
        'fold'        : i + 1,
        'train_start' : train_start,
        'train_end'   : train_end,
        'test_start'  : test_start,
        'test_end'    : test_end,
    })

folds = folds[::-1]
for i, f in enumerate(folds):
    f['fold'] = i + 1

print(f'Folds built : {len(folds)}')
print(f'Window type : {"sliding " + str(MAX_TRAIN_YEARS) + "yr" if MAX_TRAIN_YEARS else "anchored expanding"}')
print()
print(f'  {"Fold":<6} {"Train Start":<14} {"Train End":<14} '
      f'{"Test Start":<14} {"Test End":<14} {"Train Days":>10}')
print('  ' + '-' * 72)
for f in folds:
    days = (f['train_end'] - f['train_start']).days
    print(f'  {f["fold"]:<6} {str(f["train_start"].date()):<14} '
          f'{str(f["train_end"].date()):<14} '
          f'{str(f["test_start"].date()):<14} '
          f'{str(f["test_end"].date()):<14} {days:>10,}d')

## Cell 14 — Walk-Forward CV

In [ ]:
REGIME_BREAK_PERIODS = [
    (pd.Timestamp('2020-01-01'), pd.Timestamp('2021-06-30')),
    (pd.Timestamp('2022-01-01'), pd.Timestamp('2022-12-31')),
]

def get_clean_val_window(train_end, min_val_start):

    VAL_MONTHS = 6
    val_end = pd.Timestamp(train_end) - pd.Timedelta(days=1)

    for rb_start, rb_end in REGIME_BREAK_PERIODS:
        if rb_start <= val_end <= rb_end:
            val_end = rb_start - pd.Timedelta(days=1)
            break

    val_start = val_end - relativedelta(months=VAL_MONTHS)

    if val_start < min_val_start:
        val_start = min_val_start

    return val_start, val_end

cv_results = []

for fold in folds:
    fn = fold['fold']
    print(f'\n{"─"*72}')
    print(f'Fold {fn}/{len(folds)} | '
          f'train {fold["train_start"].date()} -> {fold["train_end"].date()} | '
          f'test  {fold["test_start"].date()} -> {fold["test_end"].date()}')
    print('─' * 72)

    mask_tr = ((df_lstm[DATE_COL] >= fold['train_start']) &
               (df_lstm[DATE_COL] <  fold['train_end']))
    mask_te = ((df_lstm[DATE_COL] >= fold['test_start']) &
               (df_lstm[DATE_COL] <  fold['test_end']))

    df_tr_raw = df_lstm[mask_tr].copy()
    df_te_raw = df_lstm[mask_te].copy()

    if df_tr_raw.empty or df_te_raw.empty:
        print('  [SKIP] Empty train or test slice.')
        continue

    df_tr_imp = impute_features(
        df_tr_raw.sort_values([TICKER_COL, DATE_COL]),
        LSTM_FEATURE_COLS, FILL_VALUES, DEFAULT_FILL
    )
    del df_tr_raw

    fold_norm = CrossSectionalNormalizer(LSTM_FEATURE_COLS, min_stocks=CS_MIN_STOCKS)
    fold_norm.fit(df_tr_imp)

    df_tr_norm = fold_norm.transform(df_tr_imp)
    del df_tr_imp
    gc.collect()

    tr_dates_sorted = np.sort(df_tr_norm[DATE_COL].unique())
    min_val_start   = pd.Timestamp(
        tr_dates_sorted[int(len(tr_dates_sorted) * 0.50)]
    )
    fold_val_start, fold_val_end = get_clean_val_window(
        fold['train_end'], min_val_start
    )
    print(f'  Val window : {fold_val_start.date()} -> {fold_val_end.date()}  '
          f'(regime-break clean, fixed 6mo)')

    grouped_tr = {k: v.sort_values(DATE_COL)
                  for k, v in df_tr_norm.groupby(TICKER_COL, sort=False)}
    val_parts  = []
    for ticker in tickers:
        t = grouped_tr.get(ticker)
        if t is None:
            continue
        ctx = t[t[DATE_COL] < fold_val_start].tail(SEQ_LEN).copy()
        vbt = t[(t[DATE_COL] >= fold_val_start) &
                (t[DATE_COL] <= fold_val_end)].copy()
        if vbt.empty:
            continue
        val_parts.append(pd.concat([ctx, vbt], ignore_index=True)
                           .sort_values(DATE_COL))
    del grouped_tr

    X_val_fold = y_rank_val = y_vol_val = None
    if val_parts:
        df_val_fold = (pd.concat(val_parts, ignore_index=True)
                         .sort_values([TICKER_COL, DATE_COL]))
        del val_parts
        _Xv, _yrv, _yvv, _dv = build_test_sequences(
            df_val_fold, tickers, LSTM_FEATURE_COLS, SEQ_LEN
        )
        del df_val_fold
        if _Xv is not None and len(_Xv) > 0:
            val_date_set = set(pd.to_datetime(
                tr_dates_sorted[
                    (tr_dates_sorted >= fold_val_start.to_datetime64()) &
                    (tr_dates_sorted <= fold_val_end.to_datetime64())
                ]
            ))
            vm = np.array([pd.Timestamp(d) in val_date_set for d in _dv])
            if vm.any():
                X_val_fold = _Xv[vm]
                y_rank_val = _yrv[vm]
                y_vol_val  = _yvv[vm]
        del _Xv, _yrv, _yvv, _dv
    else:
        del val_parts

    gc.collect()
    n_val_seq = len(X_val_fold) if X_val_fold is not None else 0
    print(f'  Val   sequences : {n_val_seq:,}')

    df_tr_train_only = df_tr_norm[df_tr_norm[DATE_COL] < fold_val_start].copy()
    del df_tr_norm
    gc.collect()

    result = build_train_dataset(
        df_tr_train_only, tickers, LSTM_FEATURE_COLS, SEQ_LEN, BATCH_SIZE
    )
    del df_tr_train_only
    gc.collect()

    if result[0] is None:
        print('  [SKIP] No train sequences built.')
        keras.backend.clear_session()
        gc.collect()
        continue

    train_ds, n_train, _X_tr, _yr_tr, _yv_tr = result
    print(f'  Train sequences : {n_train:,}')

    df_te_imp  = impute_features(
        df_te_raw.sort_values([TICKER_COL, DATE_COL]),
        LSTM_FEATURE_COLS, FILL_VALUES, DEFAULT_FILL
    )
    del df_te_raw
    df_te_norm = fold_norm.transform(df_te_imp)
    del df_te_imp
    gc.collect()

    X_te, y_rank_te, y_vol_te, dates_te = build_test_sequences(
        df_te_norm, tickers, LSTM_FEATURE_COLS, SEQ_LEN
    )
    del df_te_norm
    gc.collect()

    if X_te is None:
        print('  [SKIP] No test sequences built.')
        del train_ds
        del _X_tr, _yr_tr, _yv_tr
        gc.collect()
        keras.backend.clear_session()
        gc.collect()
        continue

    print(f'  Test  sequences : {len(X_te):,}  shape={X_te.shape}')

    keras.backend.clear_session()
    gc.collect()

    fold_model = build_lstm_model(
        SEQ_LEN, len(LSTM_FEATURE_COLS),
        LEARNING_RATE, LOSS_WEIGHT_RANK, LOSS_WEIGHT_VOL
    )
    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE,
        mode='min', restore_best_weights=True, verbose=0
    )
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6, verbose=0
    )

    val_data = (
        (X_val_fold, {'rank_output': y_rank_val, 'vol_output': y_vol_val})
        if X_val_fold is not None and len(X_val_fold) > 0
        else (X_te, {'rank_output': y_rank_te, 'vol_output': y_vol_te})
    )

    history = fold_model.fit(
        train_ds,
        validation_data=val_data,
        epochs=EPOCHS,
        callbacks=[early_stop, reduce_lr],
        verbose=0
    )
    epochs_run = len(history.history['loss'])

    del train_ds
    del _X_tr, _yr_tr, _yv_tr
    if X_val_fold is not None:
        del X_val_fold, y_rank_val, y_vol_val
    gc.collect()

    rank_pred, vol_pred = fold_model.predict(X_te, verbose=0)
    ic_rank, icir_rank, _ = compute_ic_cross_sectional(
        y_rank_te, rank_pred.flatten(), dates_te
    )
    ic_vol = compute_ic_pooled(y_vol_te, vol_pred.flatten())

    cv_results.append({
        'fold'      : fn,
        'ic_rank'   : ic_rank,
        'icir_rank' : icir_rank,
        'ic_vol'    : ic_vol,
        'epochs_run': epochs_run,
        'n_train'   : n_train,
        'n_test'    : len(X_te),
        'history'   : history.history,
    })

    print(f'  Epochs run   : {epochs_run}')
    print(f'  IC Rank (H1) : {ic_rank:.4f}  ICIR={icir_rank:.2f}  [cross-sectional]')
    print(f'  IC Vol  (H2) : {ic_vol:.4f}  [pooled]')

    del fold_model, X_te, y_rank_te, y_vol_te, rank_pred, vol_pred, history
    gc.collect()
    keras.backend.clear_session()
    gc.collect()

print('\nWalk-forward CV complete.')

## Cell 15 — CV Summary

In [ ]:
ic_ranks   = [r['ic_rank']   for r in cv_results]
icir_ranks = [r['icir_rank'] for r in cv_results]
ic_vols    = [r['ic_vol']    for r in cv_results]

print('=' * 72)
print('WALK-FORWARD CV RESULTS — LSTM DUAL-HEAD v4.1')
print('=' * 72)
print(f'{"Fold":<6} {"IC Rank (H1)":>14} {"ICIR (H1)":>11} '
      f'{"IC Vol (H2)":>13} {"Epochs":>8}')
print('─' * 56)
for r in cv_results:
    print(f'{r["fold"]:<6} {r["ic_rank"]:>14.4f} {r["icir_rank"]:>11.2f} '
          f'{r["ic_vol"]:>13.4f} {r["epochs_run"]:>8}')
print('─' * 56)
print(f'{"Mean":<6} {np.mean(ic_ranks):>14.4f} {np.mean(icir_ranks):>11.2f} '
      f'{np.mean(ic_vols):>13.4f}')
print(f'{"Std":<6} {np.std(ic_ranks):>14.4f} {np.std(icir_ranks):>11.2f} '
      f'{np.std(ic_vols):>13.4f}')
print(f'{"Min":<6} {np.min(ic_ranks):>14.4f} {np.min(icir_ranks):>11.2f} '
      f'{np.min(ic_vols):>13.4f}')
print(f'{"Max":<6} {np.max(ic_ranks):>14.4f} {np.max(icir_ranks):>11.2f} '
      f'{np.max(ic_vols):>13.4f}')
print('=' * 72)

mean_ic   = np.mean(ic_ranks)
mean_icir = np.mean(icir_ranks)
if   mean_ic > 0.06: grade = 'STRONG (IC > 0.06)'
elif mean_ic > 0.04: grade = 'GOOD (IC 0.04-0.06)'
elif mean_ic > 0.02: grade = 'MODERATE (IC 0.02-0.04)'
else:                grade = 'WEAK (IC < 0.02) — review features or extend window'
print(f'\nHead 1 IC grade  : {grade}')
print(f'Head 1 mean ICIR : {mean_icir:.2f}  (> 0.5 = usable, > 1.0 = strong)')

## Cell 16 — CV Plots

In [ ]:
fold_nums = [r['fold'] for r in cv_results]
fig, axes = plt.subplots(1, 3, figsize=(21, 5))

ax = axes[0]
x, w = np.arange(len(fold_nums)), 0.35
ax.bar(x - w/2, ic_ranks, w, label='Head 1 - Rank IC', color='#2980b9',
       edgecolor='black', linewidth=0.5)
ax.bar(x + w/2, ic_vols,  w, label='Head 2 - Vol IC',  color='#e67e22',
       edgecolor='black', linewidth=0.5)
ax.axhline(np.mean(ic_ranks), color='#2980b9', linestyle='--', linewidth=1,
           alpha=0.8, label=f'Rank mean ({np.mean(ic_ranks):.3f})')
ax.axhline(np.mean(ic_vols), color='#e67e22', linestyle='--', linewidth=1,
           alpha=0.8, label=f'Vol mean ({np.mean(ic_vols):.3f})')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Fold'); ax.set_ylabel('IC (Spearman)')
ax.set_title('Walk-Forward CV — IC per Fold')
ax.set_xticks(x); ax.set_xticklabels([f'F{n}' for n in fold_nums])
ax.legend(fontsize=8)

ax = axes[1]
colors = ['#27ae60' if v >= 0.5 else ('#f39c12' if v >= 0 else '#e74c3c')
          for v in icir_ranks]
ax.bar(np.arange(len(fold_nums)), icir_ranks,
       color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(0.5, color='#27ae60', linestyle='--', linewidth=1, label='ICIR=0.5')
ax.axhline(1.0, color='#2980b9', linestyle='--', linewidth=1, label='ICIR=1.0')
ax.axhline(0,   color='black', linewidth=0.8)
ax.set_xlabel('Fold'); ax.set_ylabel('ICIR')
ax.set_title('Walk-Forward CV — Rank Head ICIR')
ax.set_xticks(np.arange(len(fold_nums)))
ax.set_xticklabels([f'F{n}' for n in fold_nums])
ax.legend(fontsize=8)

ax = axes[2]
last = cv_results[-1]
ax.plot(last['history']['loss'],     label='Train loss', color='#2980b9')
ax.plot(last['history']['val_loss'], label='Val loss',   color='#e74c3c')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title(f'Training Loss — Fold {last["fold"]} (most recent)')
ax.legend()

plt.tight_layout()
cv_plot_path = os.path.join(RESULTS_DIR, f'lstm_cv_results_v4_{DATE_STAMP}.png')
plt.savefig(cv_plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {cv_plot_path}')

## Cell 17 — Fit Final Normalizer on Full Training Pool

In [ ]:
print('Fitting final cross-sectional normalizer on full training pool ...')

df_lstm_imp = impute_features(
    df_lstm.sort_values([TICKER_COL, DATE_COL]),
    LSTM_FEATURE_COLS, FILL_VALUES, DEFAULT_FILL
)

final_norm = CrossSectionalNormalizer(LSTM_FEATURE_COLS, min_stocks=CS_MIN_STOCKS)
final_norm.fit(df_lstm_imp)
print('Final normalizer fitted.')

## Cell 18 — Build Final Training + Validation Datasets [FIX-VAR + FIX-VAL]

In [ ]:
print('Normalizing full training pool ...')
df_lstm_norm = final_norm.transform(df_lstm_imp)
del df_lstm_imp
gc.collect()
print(f'Normalized rows : {len(df_lstm_norm):,}')

print('Building final train dataset via generator (FIX-OOM) ...')

grouped_final = {
    k: v.sort_values(DATE_COL).reset_index(drop=True)
    for k, v in df_lstm_norm.groupby(TICKER_COL, sort=False)
}

seq_index = []
for ticker in tickers:
    t_df = grouped_final.get(ticker)
    if t_df is None or len(t_df) < SEQ_LEN + 1:
        continue
    feat_vals = t_df[LSTM_FEATURE_COLS].values.astype(np.float32)
    for i in range(SEQ_LEN, len(t_df)):
        r = t_df.at[i, TARGET_RANK]
        v = t_df.at[i, TARGET_VOL]
        if pd.isna(r) or pd.isna(v):
            continue
        seq_index.append((ticker, i, float(r), float(v)))

n_final = len(seq_index)
print(f'Total train sequences : {n_final:,}')

if n_final == 0:
    raise RuntimeError('No sequences built from full training pool.')

idx_arr = np.random.permutation(n_final).astype(np.int32)
n_feat  = len(LSTM_FEATURE_COLS)

feat_cache = {
    ticker: grouped_final[ticker][LSTM_FEATURE_COLS].values.astype(np.float32)
    for ticker in tickers if ticker in grouped_final
}
del grouped_final, df_lstm_norm
gc.collect()

def final_generator():
    for idx in idx_arr:
        ticker, i, r, v = seq_index[idx]
        feat = feat_cache[ticker]
        yield feat[i - SEQ_LEN:i], {'rank_output': r, 'vol_output': v}

final_train_ds = (
    tf.data.Dataset.from_generator(
        final_generator,
        output_signature=(
            tf.TensorSpec(shape=(SEQ_LEN, n_feat), dtype=tf.float32),
            {
                'rank_output': tf.TensorSpec(shape=(), dtype=tf.float32),
                'vol_output' : tf.TensorSpec(shape=(), dtype=tf.float32),
            }
        )
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

all_dates_sorted = np.sort(df_lstm[DATE_COL].unique())
n_val_dates      = max(1, int(len(all_dates_sorted) * 0.10))
val_start        = pd.Timestamp(all_dates_sorted[-n_val_dates])

for rb_start, rb_end in REGIME_BREAK_PERIODS:
    if not (val_start > rb_end or
            pd.Timestamp(all_dates_sorted[-1]) < rb_start):
        val_start = rb_start - pd.Timedelta(days=180)
        break

grouped_lstm_val = {k: v.sort_values(DATE_COL)
                    for k, v in df_lstm.groupby(TICKER_COL, sort=False)}
val_parts = []
for ticker in tickers:
    t_full = grouped_lstm_val.get(ticker)
    if t_full is None:
        continue
    ctx = t_full[t_full[DATE_COL] < val_start].tail(SEQ_LEN).copy()
    bt  = t_full[t_full[DATE_COL] >= val_start].copy()
    if bt.empty:
        continue
    val_parts.append(pd.concat([ctx, bt], ignore_index=True)
                       .sort_values(DATE_COL))
del grouped_lstm_val

if not val_parts:
    raise RuntimeError('No final validation parts built.')

df_val_combined = (pd.concat(val_parts, ignore_index=True)
                     .sort_values([TICKER_COL, DATE_COL]))
del val_parts
gc.collect()

df_val_imp  = impute_features(df_val_combined, LSTM_FEATURE_COLS,
                               FILL_VALUES, DEFAULT_FILL)
del df_val_combined
df_val_norm = final_norm.transform(df_val_imp)
del df_val_imp

X_val_f, y_rank_vf, y_vol_vf, dates_val_f = build_test_sequences(
    df_val_norm, tickers, LSTM_FEATURE_COLS, SEQ_LEN
)
del df_val_norm
gc.collect()

val_date_set = set(pd.to_datetime(all_dates_sorted[-n_val_dates:]))
val_mask     = np.array([pd.Timestamp(d) in val_date_set for d in dates_val_f])
X_val_f      = X_val_f[val_mask]
y_rank_vf    = y_rank_vf[val_mask]
y_vol_vf     = y_vol_vf[val_mask]

print(f'Val sequences   : {len(X_val_f):,}  '
      f'(from {val_start.date()} — regime-break clean)')
print(f'Peak RAM used by sequence index: ~{n_final * 40 / 1e9:.2f} GB  '
      f'(index only — no full X_arr materialized)')

## Cell 19 — Train Final Model

In [ ]:
keras.backend.clear_session()
gc.collect()
print('Training final LSTM model ...')

lstm_final = build_lstm_model(
    SEQ_LEN, len(LSTM_FEATURE_COLS),
    LEARNING_RATE, LOSS_WEIGHT_RANK, LOSS_WEIGHT_VOL
)
early_stop_final = callbacks.EarlyStopping(
    monitor='val_loss', patience=PATIENCE,
    mode='min', restore_best_weights=True, verbose=1
)
reduce_lr_final = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6, verbose=1
)

final_history = lstm_final.fit(
    final_train_ds,
    validation_data=(
        X_val_f, {'rank_output': y_rank_vf, 'vol_output': y_vol_vf}
    ),
    epochs=40,
    callbacks=[early_stop_final, reduce_lr_final],
    verbose=1
)

del final_train_ds, seq_index, feat_cache, idx_arr
del X_val_f, y_rank_vf, y_vol_vf
gc.collect()
print('Final model training complete.')

## Cell 20 — Save Model and Normalizer

In [ ]:
model_path = os.path.join(MODELS_DIR, f'lstm_v4_{DATE_STAMP}.keras')
norm_path  = os.path.join(MODELS_DIR, f'lstm_norm_v4_{DATE_STAMP}.pkl')

lstm_final.save(model_path)
final_norm.save(norm_path)

print(f'Model saved     : {model_path}')
print(f'Normalizer saved: {norm_path}')
print(f'Features        : {len(LSTM_FEATURE_COLS)} columns')
print(f'Feature order   : {LSTM_FEATURE_COLS}')

## Cell 21 — Sanity Checks

In [ ]:
print('=' * 60)
print('SANITY CHECKS')
print('=' * 60)

checks = []
mean_h1 = np.mean(ic_ranks)
checks.append(('PASS' if mean_h1 > 0 else 'FAIL',
               f'Head 1 mean cross-sec. IC > 0 : {mean_h1:.4f}'))
mean_h2 = np.mean(ic_vols)
checks.append(('PASS' if mean_h2 > 0 else 'FAIL',
               f'Head 2 mean pooled IC > 0     : {mean_h2:.4f}'))
min_ic = np.min(ic_ranks)
checks.append(('PASS' if min_ic >= -0.05 else 'WARN',
               f'No fold Rank IC < -0.05        : min={min_ic:.4f}'))
checks.append(('PASS' if np.mean(icir_ranks) > 0 else 'WARN',
               f'Mean ICIR > 0                  : {np.mean(icir_ranks):.3f}'))

_x_test = np.zeros((20, SEQ_LEN, len(LSTM_FEATURE_COLS)), dtype=np.float32)
_r, _v  = lstm_final.predict(_x_test, verbose=0)
checks.append(('PASS' if _r.shape == (20, 1) and _v.shape == (20, 1) else 'FAIL',
               f'Output shapes correct           : rank={_r.shape}, vol={_v.shape}'))
checks.append(('PASS' if _v.min() > 0 else 'FAIL',
               f'Vol predictions positive        : min={_v.min():.4f}'))
_r_mean = float(_r.mean())
checks.append(('PASS' if -0.05 < _r_mean < 0.05 else 'WARN',
               f'Return predictions near 0        : mean={_r_mean:.4f}'))

leak_rows = df_lstm[df_lstm[DATE_COL] >= BACKTEST_START]
checks.append(('PASS' if len(leak_rows) == 0 else 'FAIL',
               f'No backtest dates in train pool : {len(leak_rows)} rows'))

for path, label in [(model_path, 'Model'), (norm_path, 'Normalizer')]:
    checks.append(('PASS' if os.path.exists(path) else 'FAIL',
                   f'File exists: {label} ({os.path.basename(path)})'))

norm_dates  = final_norm.stats_.index
future_norm = norm_dates[norm_dates > TRAIN_END_DATE]
checks.append(('PASS' if len(future_norm) == 0 else 'FAIL',
               f'Normalizer no post-cutoff dates : {len(future_norm)} found'))

for flag, msg in checks:
    print(f'[{flag}] {msg}')

print('\n' + '=' * 60)
print('ALL CHECKS COMPLETE')
print('=' * 60)

## Cell 22 — Backtest Evaluation (2024-Jan → Present)

In [ ]:
print('\n' + '=' * 60)
print('BACKTEST EVALUATION — 2024-Jan -> Present')
print('=' * 60)

grouped_lstm_ctx = {k: v.sort_values(DATE_COL)
                    for k, v in df_lstm.groupby(TICKER_COL, sort=False)}
grouped_backtest = {k: v.sort_values(DATE_COL)
                    for k, v in df_backtest.groupby(TICKER_COL, sort=False)}

bt_parts = []
for ticker in tickers:
    ctx_df = grouped_lstm_ctx.get(ticker)
    bt_df  = grouped_backtest.get(ticker)
    if ctx_df is None or bt_df is None or bt_df.empty:
        continue
    ctx = ctx_df.tail(SEQ_LEN).copy()
    bt_parts.append(
        pd.concat([ctx, bt_df], ignore_index=True).sort_values(DATE_COL)
    )

del grouped_lstm_ctx, grouped_backtest

if not bt_parts:
    print('[WARN] No tickers found in backtest range.')
else:
    df_bt_combined = (pd.concat(bt_parts, ignore_index=True)
                        .sort_values([TICKER_COL, DATE_COL]))
    del bt_parts
    gc.collect()

    df_bt_imp  = impute_features(df_bt_combined, LSTM_FEATURE_COLS,
                                  FILL_VALUES, DEFAULT_FILL)
    del df_bt_combined
    df_bt_norm = final_norm.transform(df_bt_imp)
    del df_bt_imp
    gc.collect()

    X_bt, y_rank_bt, y_vol_bt, dates_bt = build_test_sequences(
        df_bt_norm, tickers, LSTM_FEATURE_COLS, SEQ_LEN
    )
    del df_bt_norm
    gc.collect()

    if X_bt is not None:
        bt_mask   = np.array([pd.Timestamp(d) >= BACKTEST_START for d in dates_bt])
        X_bt      = X_bt[bt_mask]
        y_rank_bt = y_rank_bt[bt_mask]
        y_vol_bt  = y_vol_bt[bt_mask]
        dates_bt  = dates_bt[bt_mask]

    if X_bt is None or len(X_bt) == 0:
        print('[WARN] No backtest sequences built.')
        print('       Expected if Target_Rank_21d is NULL for recent dates.')
    else:
        print(f'Backtest sequences : {len(X_bt):,}  shape={X_bt.shape}')

        rank_bt_pred, vol_bt_pred = lstm_final.predict(
            X_bt, batch_size=BATCH_SIZE, verbose=0
        )
        rank_bt_pred = rank_bt_pred.flatten()
        vol_bt_pred  = vol_bt_pred.flatten()

        bt_ic, bt_icir, bt_ic_series = compute_ic_cross_sectional(
            y_rank_bt, rank_bt_pred, dates_bt
        )
        bt_vol_ic = compute_ic_pooled(y_vol_bt, vol_bt_pred)

        print(f'\n{"-"*50}')
        print(f'  Backtest Rank IC  (cross-sec.) : {bt_ic:.4f}')
        print(f'  Backtest Rank ICIR             : {bt_icir:.2f}')
        print(f'  Backtest Vol  IC  (pooled)     : {bt_vol_ic:.4f}')
        print(f'  Backtest trading days          : {len(bt_ic_series):,}')
        print(f'{"-"*50}')

        cv_mean = np.mean(ic_ranks)
        if bt_ic > cv_mean * 2.0 and bt_ic > 0.05:
            print(f'[WARN] Backtest IC >> CV mean — investigate leakage.')
        elif bt_ic < 0:
            print(f'[WARN] Negative backtest IC — regime shift or model drift.')

        if len(bt_ic_series) > 0:
            bt_df         = bt_ic_series.reset_index()
            bt_df.columns = ['Date', 'IC']
            bt_df['Date'] = pd.to_datetime(bt_df['Date'])
            monthly_ic    = bt_df.assign(Month=bt_df['Date'].dt.to_period('M')) \
                                  .groupby('Month')['IC'].mean()

            print('\nMonthly IC (Rank Head):')
            print(f'  {"Month":<12}  {"Mean IC":>8}  Signal')
            print(f'  {"-"*32}')
            for period, ic_val in monthly_ic.items():
                print(f'  {str(period):<12}  {ic_val:>8.4f}  {"up" if ic_val > 0 else "down"}')
            print(f'  {"-"*32}')
            print(f'  {"Overall":<12}  {monthly_ic.mean():>8.4f}')

        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        ax = axes[0]
        if len(bt_ic_series) > 0:
            cum = bt_ic_series.cumsum()
            ax.plot(cum.index, cum.values, color='#2980b9', linewidth=1.5)
            ax.axhline(0, color='black', linewidth=0.8)
            ax.fill_between(cum.index, 0, cum.values,
                            where=(cum.values >= 0), alpha=0.15, color='#27ae60')
            ax.fill_between(cum.index, 0, cum.values,
                            where=(cum.values < 0), alpha=0.15, color='#e74c3c')
        ax.set_title('Backtest — Cumulative IC'); ax.set_xlabel('Date')

        ax = axes[1]
        if len(bt_ic_series) > 0:
            mc = ['#27ae60' if v >= 0 else '#e74c3c' for v in monthly_ic.values]
            ax.bar(range(len(monthly_ic)), monthly_ic.values,
                   color=mc, edgecolor='black', linewidth=0.5)
            ax.set_xticks(range(len(monthly_ic)))
            ax.set_xticklabels([str(p) for p in monthly_ic.index],
                               rotation=45, ha='right')
            ax.axhline(0, color='black', linewidth=0.8)
            ax.set_title('Backtest — Monthly IC')

        plt.tight_layout()
        bt_plot = os.path.join(RESULTS_DIR, f'lstm_backtest_v42_{DATE_STAMP}.png')
        plt.savefig(bt_plot, dpi=150, bbox_inches='tight')
        plt.close()
        print(f'\nBacktest plot saved: {bt_plot}')

        if   bt_ic > 0.06: bt_grade = 'STRONG'
        elif bt_ic > 0.04: bt_grade = 'GOOD'
        elif bt_ic > 0.02: bt_grade = 'MODERATE — usable, some regime shift'
        elif bt_ic > 0:    bt_grade = 'WEAK but positive — monitor live'
        else:              bt_grade = 'NEGATIVE — regime shift; retrain or review features'
        print(f'Backtest grade : {bt_grade}')

## Cell 23 — Final Summary

In [ ]:
print('\n' + '=' * 60)
print('ALL DONE — LSTM ')
print('=' * 60)
print(f'Features used        : {len(LSTM_FEATURE_COLS)}')
print(f'CV mean Rank IC      : {np.mean(ic_ranks):.4f}  [cross-sectional]')
print(f'CV mean Rank ICIR    : {np.mean(icir_ranks):.2f}')
if 'bt_ic' in dir():
    print(f'BT Rank IC           : {bt_ic:.4f}  [held-out 2024+]')
    print(f'BT Rank ICIR         : {bt_icir:.2f}')
print(f'Model saved          : {model_path}')
print(f'Normalizer saved     : {norm_path}')




In [ ]:
import shutil
shutil.make_archive('/kaggle/working/lstm_models', 'zip',
                    '/kaggle/working/models')
print('Done — download lstm_models.zip from Output panel')